# 10 — Thực nghiệm cuối trên test: GRU4Rec gốc và GRU4Rec đa hành vi, phiên 30 và 60 phút

Phạm vi theo protocol v1.3.2, gồm cả lần đọc test này. Người dùng duyệt lần đọc test cho phạm vi v1.3.0 ngày 24/11/2026; ngày 26/11/2026 người dùng tạm bỏ SR-GNN (v1.3.1), dừng và xóa lần chạy test v1.3.0 đang dở, và duyệt chạy lại test một lần với phạm vi mới. Vì vậy test đã được đọc một phần trước lần chạy này. Cùng ngày, v1.3.2 bỏ S-POP, SR và V-SKNN; lần chạy test v1.3.1 bị chặn trước khi đọc test. Mọi cấu hình đã được chọn trên validation (notebook 05–09) và được đóng băng trước khi đọc test. Test chỉ chạy **một lần**.

- **Bài toán:** tương tác kế tiếp. Target là tập item phân biệt của nhóm timestamp kế tiếp, mọi hành vi. Mẫu số gồm cả query có target ngoài vocabulary, tính 0 điểm.
- **Mô hình:** `gru4rec_item` (GRU4Rec gốc), `gru4rec` (GRU4Rec đa hành vi), cùng các mốc Popularity, Repeat-last và VSTAN.
- **Hai ngưỡng phiên:** mỗi ngưỡng có dữ liệu fit, vocabulary, `known_at` và cấu hình riêng.
- **Hai bộ query đánh giá:** mỗi mô hình được chấm trên `q30` (query test theo phiên 30 phút) và `q60` (query test theo phiên 60 phút). So sánh hai ngưỡng luôn làm **trên cùng một bộ query**, có paired bootstrap.
- **Hai biến thể fit (D2):**
  - `train_only`: mô hình đã chọn, fit trên train;
  - `train_validation_refit`: **biến thể chính**, fit lại trên train + validation với đúng cấu hình và số epoch đã chọn.
- **Diễn tập:** đặt `RR_FINAL_SPLIT=validation` để chạy toàn bộ đường code trên validation, ghi vào `reports/10_final_rehearsal/`. Ở chế độ này, biến thể refit chỉ fit trên train, mỗi mô hình neural 1 epoch. Lần chạy thật dùng mặc định `test` và chỉ chạy được khi đã có bản diễn tập với **đúng** cấu hình đóng băng hiện tại.

## 1. Chế độ chạy và đóng băng cấu hình

In [ ]:
from pathlib import Path
import os, sys, time

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src/rr").is_dir())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from rr.io import environment, fingerprint, read_json, rel, report_path, require, save_verified, utc_now, verify_inputs, write_json
from rr.ablation import score_models, topk_matrix
from rr.metrics import bootstrap_ci, paired_bootstrap
from rr.samples import build_vocab

SPLIT = os.environ.get("RR_FINAL_SPLIT", "test")
require(SPLIT in {"test", "validation"}, "RR_FINAL_SPLIT phải là test hoặc validation.")
REHEARSAL = SPLIT == "validation"
OUT = ROOT / ("reports/10_final_rehearsal" if REHEARSAL else "reports/10_final")
PROTOCOL_PATH = ROOT / "configs/protocol.json"
PROTOCOL = read_json(PROTOCOL_PATH)
require(PROTOCOL["version"] == "1.3.2", "Notebook này viết cho protocol v1.3.2.")
PROCESSED = ROOT / "data/processed"
DATA60 = PROCESSED / "session60"
SEED = PROTOCOL["seed"]
BOOT, PAIRED = PROTOCOL["evaluation"]["bootstrap"], PROTOCOL["evaluation"]["paired_bootstrap"]
WIDTH = 200
MODELS = ["popularity", "repeat_last", "vstan", "gru4rec_item", "gru4rec"]
NAMES = {"popularity": "Popularity", "repeat_last": "Repeat-last", "vstan": "VSTAN",
         "gru4rec_item": "GRU4Rec gốc (chỉ item)", "gru4rec": "GRU4Rec đa hành vi"}
require(set(MODELS) == set(PROTOCOL["scope"]["reference_models"]) | set(PROTOCOL["scope"]["focus_models"]), "Danh sách mô hình khác protocol.")
TCOLS = ["query_id", "itemid", "target_is_repeat"]
torch.set_num_threads(4)

LOG = os.environ.get("RR_PROGRESS_LOG")
def log(msg):
    line = f"{pd.Timestamp.now(tz='UTC'):%H:%M:%S} [10 {SPLIT}] {msg}"
    print(line, flush=True)
    if LOG:
        with open(LOG, "a", encoding="utf-8") as f:
            f.write(line + "\n")

val = read_json(ROOT / "reports/09_validation_30_60/metrics.json")
EVALREP = {n: read_json(ROOT / f"reports/05_08_validation/{n}_validation.json") for n in ["popularity", "knn", "gru4rec"]}
MODEL_FILES = [f"models/{d}/{f}" for d in ["gru4rec", "gru4rec_item", "session60/gru4rec", "session60/gru4rec_item"]
               for f in ["model.pt", "config.json"]]
FROZEN_FILES = ["configs/protocol.json", "reports/09_validation_30_60/metrics.json",
                *[f"reports/05_08_validation/{n}_validation.json" for n in EVALREP], *MODEL_FILES]
frozen = {p: fingerprint(ROOT / p) for p in FROZEN_FILES}
require(val["protocol"]["sha256"] == frozen["configs/protocol.json"]["sha256"], "protocol.json đổi sau notebook 09.")
for p in MODEL_FILES:
    if p in val["artifacts"]:
        require(frozen[p] == val["artifacts"][p], f"{p} khác lúc notebook 09 lưu.")
require(frozen["models/gru4rec/model.pt"]["sha256"] == EVALREP["gru4rec"]["model_file"]["sha256"], "models/gru4rec/model.pt khác notebook 07.")
for n in EVALREP:
    require(frozen[f"reports/05_08_validation/{n}_validation.json"] == val["inputs"][f"reports/05_08_validation/{n}_validation.json"], f"Report {n} đổi sau notebook 09.")
if not REHEARSAL:
    require(not (OUT / "metrics.json").exists(), "Test đã được chạy; không chạy lại (test chỉ đọc một lần).")
    reh = ROOT / "reports/10_final_rehearsal/metrics.json"
    require(reh.exists() and read_json(reh)["frozen"] == frozen, "Cần bản diễn tập (RR_FINAL_SPLIT=validation) với đúng cấu hình đóng băng hiện tại.")
OUT.mkdir(parents=True, exist_ok=True)
write_json(OUT / "frozen.json", {"generated_at_utc": utc_now(), "split": SPLIT, "files": frozen})
log("đã đóng băng cấu hình")

## 2. Kiểm tra đầu vào và ví dụ biên

Đầu vào 30 phút khớp SHA-256 trong report của notebook 02, 03 và 04; đầu vào 60 phút khớp report của notebook 09.

Ví dụ biên:
- Top-K dạng dài chuyển đúng thành ma trận.
- Paired bootstrap của hai danh sách giống nhau cho hiệu bằng 0.
- Ở mỗi ngưỡng, hai biến thể GRU4Rec có cùng vocabulary, chỉ khác `use_behavior`.

In [ ]:
prev = {n: read_json(report_path(ROOT, n)) for n in ["session_split", "metadata", "samples"]}
expected = {p: v for p, v in val["artifacts"].items() if p.startswith("data/")}
expected["data/processed/events_sessionized.parquet"] = prev["session_split"]["artifacts"]["data/processed/events_sessionized.parquet"]
for p in ["item_property_history", "item_catalog"]:
    expected[f"data/processed/{p}.parquet"] = prev["metadata"]["artifacts"][f"data/processed/{p}.parquet"]
expected.update(prev["samples"]["artifacts"])
inputs = verify_inputs(ROOT, {k: {"bytes": v["bytes"], "sha256": v["sha256"]} for k, v in expected.items()})
print("Đầu vào khớp SHA-256:", len(inputs), "tệp")

_m = topk_matrix(pd.DataFrame({"query_id": [3, 3], "rank": [1, 2], "itemid": [8, 9]}), [3, 4], 3)
assert _m.tolist() == [[8, 9, -1], [-1, -1, -1]]
assert paired_bootstrap([0.5, 0.2], [0.5, 0.2], 50, 0)["mean_diff"] == 0
for d in ["models", "models/session60"]:
    a, b = read_json(ROOT / d / "gru4rec_item/config.json"), read_json(ROOT / d / "gru4rec/config.json")
    assert a["use_behavior"] is False and b.get("use_behavior", True) is True and a["n_items"] == b["n_items"]
    assert {k for k in a if k not in {"name", "use_behavior", "epochs", "vocabulary"} and a[k] != b.get(k)} == set()
print("Đã kiểm tra ví dụ biên.")

## 3. Dữ liệu và cấu hình của hai ngưỡng

In [ ]:
history = pd.read_parquet(PROCESSED / "item_property_history.parquet")
cols = ["session_id", "group_index", "itemid", "event", "timestamp", "split"]
TH = {}
for th, base, known_path in [("30", PROCESSED, PROCESSED / "item_catalog.parquet"), ("60", DATA60, DATA60 / "item_known_at.parquet")]:
    ev = pd.read_parquet(base / "events_sessionized.parquet", columns=cols)
    ev["split"] = ev["split"].astype(str)
    mdir = ROOT / ("models" if th == "30" else "models/session60")
    sel = val["selected"][th]
    TH[th] = {"events": ev, "known": pd.read_parquet(known_path), "base": base,
              "vocab_train": pd.read_parquet(base / "item_vocab.parquet"),
              "cfg": {"popularity": sel["popularity"], "knn_selected": sel["knn"], "models": MODELS,
                      "gru_variants": {n: read_json(mdir / n / "config.json") for n in ["gru4rec_item", "gru4rec"]},
                      "model_dir": mdir,
                      "fallback_top_n": PROTOCOL["candidates"]["fallback_top_n"], "max_groups": PROTOCOL["knn"]["max_prefix_groups"],
                      "max_len": PROTOCOL["long_sessions"]["gru_train_max_prefix_groups"] + 1, "seed": SEED, "width": WIDTH, "root": ROOT}}
    for n in ["gru4rec_item", "gru4rec"]:
        require(TH[th]["cfg"]["gru_variants"][n]["epochs"] == sel[f"{n}_epochs"], f"{th} phút {n}: số epoch khác notebook 09.")
EVAL = {f"q{th}": (pd.read_parquet(TH[th]["base"] / f"samples/{SPLIT}_queries.parquet"),
                   pd.read_parquet(TH[th]["base"] / f"samples/{SPLIT}_targets.parquet", columns=TCOLS),
                   TH[th]["events"][TH[th]["events"]["split"] == SPLIT].reset_index(drop=True)) for th in ["30", "60"]}
for name, (q, t, e) in EVAL.items():
    print(f"{name}: {len(q):,} query, {len(t):,} lượt target")
display(pd.DataFrame({th: {"popularity": TH[th]["cfg"]["popularity"]["popularity_behavior_weights"],
                           "vstan": TH[th]["cfg"]["knn_selected"]["vstan"]["best_run"],
                           **{f"{n} epochs": c["epochs"] for n, c in TH[th]["cfg"]["gru_variants"].items()}} for th in TH}))

VARIANTS = {"train_only": {"fit_splits": ["train"], "known_at": "known_at_train_ms", "neural": "load"},
            "train_validation_refit": {"fit_splits": ["train", "validation"], "known_at": "known_at_trainval_ms", "neural": "train"}}
if REHEARSAL:  # exercise the refit path without leakage: refit on train only, 1 epoch
    VARIANTS["train_validation_refit"] = {"fit_splits": ["train"], "known_at": "known_at_train_ms", "neural": "train", "epochs": 1}
MAIN = "train_validation_refit"

## 4. Chạy mọi tổ hợp

Với mỗi ngưỡng phiên và mỗi biến thể fit, notebook fit năm mô hình một lần, rồi dùng đúng các mô hình đó để chấm cả `q30` và `q60`.

Top-K của biến thể `train_only` phiên 30 phút trên `q30` được lưu lại. Notebook 11 (demo) và notebook 12 (ablation dữ liệu) dùng Top-K này để đối chiếu.

In [ ]:
ALL, PER_QUERY, TIMES, TOPK = {}, [], {}, {}
for th in ["30", "60"]:
    ev = TH[th]["events"]
    for vname, vcfg in VARIANTS.items():
        t_var = time.perf_counter()
        fit_events = ev[ev["split"].isin(vcfg["fit_splits"])].reset_index(drop=True)
        cfg = dict(TH[th]["cfg"])
        if "epochs" in vcfg:
            cfg["gru_variants"] = {n: {**c, "epochs": vcfg["epochs"]} for n, c in cfg["gru_variants"].items()}
        if vname == "train_only":
            require(build_vocab(fit_events).equals(TH[th]["vocab_train"]), f"{th} phút: vocabulary train khác bản đã lưu.")
        elif th == "60" and not REHEARSAL:
            require(build_vocab(fit_events).equals(pd.read_parquet(DATA60 / "item_vocab_trainval.parquet")), "Vocabulary train + validation khác notebook 09.")
        neural_models, res_v = None, {}
        for ename, (eq, et, ee) in EVAL.items():
            log(f"fit {th} phút / {vname} / {ename}: bắt đầu")
            keep = ("vstan", "gru4rec_item", "gru4rec") if (th, vname, ename) == ("30", "train_only", "q30") else ()
            res, neural_models = score_models(fit_events, ee, eq, et, TH[th]["known"], history, vcfg["known_at"], cfg,
                                              vcfg["neural"] if neural_models is None else "reuse",
                                              neural_models=neural_models, log=log, keep_topk=keep)
            for name in keep:
                TOPK[name] = res[name].pop("topk")
            for name in MODELS:
                r = res[name]
                nd = r["per_query"]["ndcg@20"].to_numpy()
                r["report"]["bootstrap_ndcg@20"] = bootstrap_ci(nd, BOOT["n_resamples"], BOOT["seed"], BOOT["level"])
                r["report"]["paired_vs_repeat_last"] = paired_bootstrap(nd, res["repeat_last"]["per_query"]["ndcg@20"].to_numpy(),
                                                                        PAIRED["n_resamples"], PAIRED["seed"], PAIRED["level"])
                PER_QUERY.append(pd.DataFrame({"fit_threshold": th, "variant": vname, "eval_set": ename, "model": name,
                                               "query_id": r["per_query"]["query_id"].to_numpy(),
                                               "hit@20": r["per_query"]["hit@20"].to_numpy(), "ndcg@20": nd}))
            res_v[ename] = res
            log(f"fit {th} phút / {vname} / {ename}: xong | " + ", ".join(f"{n}={res[n]['report']['all']['ndcg@20']:.4f}" for n in MODELS))
        ALL.setdefault(th, {})[vname] = res_v
        TIMES[f"{th}/{vname}"] = round(time.perf_counter() - t_var, 1)
        del neural_models, fit_events

## 5. So sánh

1. **Hiệu ứng hành vi:** `gru4rec` − `gru4rec_item`, NDCG@20, paired bootstrap trên cùng query. Tính cho mọi tổ hợp ngưỡng × biến thể × bộ query.
2. **Hiệu ứng ngưỡng phiên:** mô hình 60 phút − mô hình 30 phút trên **cùng** bộ query (`q30` và `q60`), cho từng mô hình.
3. **So với Repeat-last:** đã có trong report của từng mô hình (`paired_vs_repeat_last`).

In [ ]:
pq_all = pd.concat(PER_QUERY, ignore_index=True)
def per_query(th, vname, ename, model):
    part = pq_all[(pq_all["fit_threshold"] == th) & (pq_all["variant"] == vname) & (pq_all["eval_set"] == ename) & (pq_all["model"] == model)]
    return part.set_index("query_id")["ndcg@20"]

def paired(a, b):
    require(a.index.equals(b.index), "Hai bên phải cùng thứ tự query.")
    return paired_bootstrap(a.to_numpy(), b.to_numpy(), PAIRED["n_resamples"], PAIRED["seed"], PAIRED["level"])

BEHAVIOR = {f"{th}/{v}/{e}": paired(per_query(th, v, e, "gru4rec"), per_query(th, v, e, "gru4rec_item"))
            for th in TH for v in VARIANTS for e in EVAL}
THRESHOLD = {f"{v}/{e}/{m}": paired(per_query("60", v, e, m), per_query("30", v, e, m)) for v in VARIANTS for e in EVAL for m in MODELS}
show = lambda d: pd.DataFrame(d).T[["mean_diff", "low", "high", "p_not_better"]]
print("GRU4Rec đa hành vi − GRU4Rec gốc"); display(show(BEHAVIOR))
print("Mô hình 60 phút − mô hình 30 phút, cùng query"); display(show(THRESHOLD))

rep = lambda th, v, e, m: ALL[th][v][e][m]["report"]
main = pd.DataFrame({(f"{th} phút (q{th})", k): {NAMES[m]: rep(th, MAIN, f"q{th}", m)["all"][k] for m in MODELS}
                     for th in TH for k in ["hit@20", "mrr@20", "ndcg@20"]})
display(main.style.format("{:.4f}"))

## 6. Lưu Top-K, kết quả từng query và báo cáo

In [ ]:
def fmt(v):
    return f"{v:.4f}" if isinstance(v, (float, np.floating)) else str(v)

def md_table(rows: dict, cols: list, first="Mô hình"):
    head = f"| {first} | " + " | ".join(cols) + " |\n|" + " --- |" * (len(cols) + 1) + "\n"
    return head + "".join(f"| {n} | " + " | ".join(fmt(v.get(c, "")) for c in cols) + " |\n" for n, v in rows.items())

ci = lambda b: f"[{b['low']:.4f}; {b['high']:.4f}]"
def results_table(vname):
    rows = {}
    for m in MODELS:
        row = {}
        for th in TH:
            r = rep(th, vname, f"q{th}", m)
            row.update({f"HR@20 ({th}')": r["all"]["hit@20"], f"MRR@20 ({th}')": r["all"]["mrr@20"],
                        f"NDCG@20 ({th}')": r["all"]["ndcg@20"], f"CI 95% ({th}')": ci(r["bootstrap_ndcg@20"])})
        rows[NAMES[m]] = row
    return rows, list(rows[NAMES[MODELS[0]]])

topk_frames = []
q30 = EVAL["q30"][0]["query_id"].to_numpy()
for name, (topk, is_fb) in TOPK.items():
    rows_, ranks = np.nonzero(topk != -1)
    topk_frames.append(pd.DataFrame({"model": name, "query_id": q30[rows_], "rank": (ranks + 1).astype(np.int16),
                                     "itemid": topk[rows_, ranks], "is_fallback": is_fb[rows_, ranks]}))
require(all(fingerprint(ROOT / p) == v for p, v in inputs.items()), "Đầu vào thay đổi trong lúc chạy.")
require(all(fingerprint(ROOT / p) == v for p, v in frozen.items()), "Cấu hình đóng băng bị thay đổi trong lúc chạy.")
topk_info = save_verified(pd.concat(topk_frames, ignore_index=True), PROCESSED / f"final/{SPLIT}_train_only_topk.parquet",
                          ["model", "query_id", "rank"], ROOT)
pq_info = save_verified(pq_all, PROCESSED / f"final/{SPLIT}_per_query.parquet", ["fit_threshold", "variant", "eval_set", "model", "query_id"], ROOT)

def slim(res):
    return {n: {"report": r["report"], "fit_seconds": r["fit_seconds"], "query_seconds": r["query_seconds"]} for n, r in res.items()}

write_json(OUT / "metrics.json", {
    "generated_at_utc": utc_now(), **environment(), "torch_version": torch.__version__,
    "notebook": "notebooks/10_thuc_nghiem_cuoi_test.ipynb", "split": SPLIT, "rehearsal": REHEARSAL,
    "approved_test_read": "user approved on 2026-11-24 (scope v1.3.0) one test read for GRU4Rec item-only vs multi-behavior, "
                          "30- and 60-minute sessions; on 2026-11-26 the user dropped SR-GNN (scope v1.3.1), stopped and deleted the "
                          "unfinished v1.3.0 test run (test already partly read) and approved this single re-run; v1.3.2 then "
                          "removed S-POP, SR and V-SKNN before any test read; references: Popularity, Repeat-last and VSTAN",
    "protocol": {"version": PROTOCOL["version"], **fingerprint(PROTOCOL_PATH)}, "scope": PROTOCOL["scope"],
    "frozen": frozen, "inputs": inputs, "n_queries": {e: int(len(q)) for e, (q, _, _) in EVAL.items()},
    "models": MODELS, "primary_variant": MAIN, "variant_seconds": TIMES,
    "results": {th: {v: {e: slim(r) for e, r in res.items()} for v, res in ALL[th].items()} for th in ALL},
    "behavior_vs_item_only": BEHAVIOR, "threshold_60_vs_30_same_queries": THRESHOLD,
    "saved_topk": topk_info, "per_query": pq_info, "status": "PASS"})

title = f"# Kết quả thực nghiệm cuối trên {SPLIT}{' (DIỄN TẬP, không dùng để báo cáo)' if REHEARSAL else ''}\n\n"
lines = [title,
         f"Sinh bởi `notebooks/10_thuc_nghiem_cuoi_test.ipynb` lúc {utc_now()}, protocol {PROTOCOL['version']}. "
         f"Bài toán: tương tác kế tiếp. `q30`: {len(EVAL['q30'][0]):,} query theo phiên 30 phút; `q60`: {len(EVAL['q60'][0]):,} query theo phiên 60 phút. "
         f"Mẫu số gồm cả query có target ngoài vocabulary. Biến thể chính: `{MAIN}`. Cột (30') là mô hình phiên 30 phút chấm trên `q30`, "
         "cột (60') là mô hình phiên 60 phút chấm trên `q60`.\n"]
for vname in [MAIN, "train_only"]:
    rows, cols_ = results_table(vname)
    lines.append(f"\n## Kết quả chính, biến thể `{vname}`\n\n" + md_table(rows, cols_))
lines.append("\n## Hiệu ứng hành vi: GRU4Rec đa hành vi − GRU4Rec gốc (NDCG@20, paired bootstrap)\n\n" + md_table(
    {k.replace("/", " · "): {"Δ": v["mean_diff"], "CI thấp": v["low"], "CI cao": v["high"], "p(không tốt hơn)": v["p_not_better"]}
     for k, v in BEHAVIOR.items()}, ["Δ", "CI thấp", "CI cao", "p(không tốt hơn)"], "Ngưỡng fit · biến thể · bộ query"))
lines.append(f"\n### Hai biến thể GRU4Rec theo loại target (`{MAIN}`, NDCG@20)\n\n" + md_table(
    {f"{NAMES[m]} ({th}')": {"mọi target": rep(th, MAIN, f"q{th}", m)["all"]["ndcg@20"],
                            "target mới": rep(th, MAIN, f"q{th}", m)["new_targets"]["ndcg@20"],
                            "target lặp": rep(th, MAIN, f"q{th}", m)["repeat_targets"]["ndcg@20"]}
     for th in TH for m in ["gru4rec_item", "gru4rec"]}, ["mọi target", "target mới", "target lặp"]))
lines.append(f"\n## Hiệu ứng ngưỡng phiên trên cùng bộ query: mô hình 60 phút − mô hình 30 phút (`{MAIN}`, NDCG@20)\n\n" + md_table(
    {NAMES[m]: {**{f"Δ {e}": THRESHOLD[f'{MAIN}/{e}/{m}']["mean_diff"] for e in EVAL},
                **{f"CI {e}": ci(THRESHOLD[f'{MAIN}/{e}/{m}']) for e in EVAL}} for m in MODELS},
    ["Δ q30", "CI q30", "Δ q60", "CI q60"]))
lines.append(f"\n## NDCG@20 theo độ dài prefix (`{MAIN}`)\n\n" + md_table(
    {f"{NAMES[m]} ({th}')": {b: v["ndcg@20"] for b, v in rep(th, MAIN, f"q{th}", m)["by_prefix_len"].items()}
     for th in TH for m in MODELS},
    ["1", "2-4", "5-9", "10+"]))
lines.append(f"\n## So với Repeat-last (`{MAIN}`, NDCG@20, paired bootstrap)\n\n" + md_table(
    {f"{NAMES[m]} ({th}')": {"Δ": rep(th, MAIN, f"q{th}", m)["paired_vs_repeat_last"]["mean_diff"],
                            "CI": ci(rep(th, MAIN, f"q{th}", m)["paired_vs_repeat_last"])}
     for th in TH for m in MODELS if m != "repeat_last"}, ["Δ", "CI"]))
lines.append("\n## Thời gian fit (giây)\n\n" + md_table(
    {NAMES[m]: {f"{th}' {v}": ALL[th][v]["q30"][m]["fit_seconds"] for th in TH for v in VARIANTS} for m in MODELS},
    [f"{th}' {v}" for th in TH for v in VARIANTS]))
(OUT / "BAO_CAO_KET_QUA.md").write_text("".join(lines), encoding="utf-8", newline="\n")
log("xong, báo cáo: " + rel(OUT / "metrics.json", ROOT))